# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL with machine-actionable metadata describing records, record sets, fields, and files via unique `@id`s.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and explore the main dataset information. You only need the dataset schema URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review the top-level structure: The dataset contains one or more `recordSet`s (tables, sheets, or text groups), each with uniquely identified fields (columns). Unique `@id`s ensure accurate programmatic selection and referencing.

The code below lists all available record sets, and for each, its fields and their `@id`s.

In [ ]:
# List all record sets, fields, and field IDs
print("RecordSets overview:")
record_sets = list(dataset.record_sets())
record_set_ids = []

for rs in record_sets:
    print(f"- RecordSet name: {rs.name} | @id: {rs.id}")
    record_set_ids.append(rs.id)

    print("   Fields (@id):")
    for f in rs.fields:
        print(f"     - {f.name} | @id: {f.id} | Data type: {getattr(f, 'data_type', 'N/A')}")
    print()

# For this demonstration, choose the first recordSet for next steps
record_set_id = record_set_ids[0] if record_set_ids else None
print(f"Default record_set_id used for extraction: {record_set_id}")

## 3. Data Extraction
You can extract the actual records (rows) from any record set by providing its `@id` (the Croissant schema unique identifier). This ensures your code always references the correct structure regardless of dataset version or structure changes.

The code below loads records from all record sets found and creates one pandas DataFrame per record set. Use the field `@id`s as DataFrame column names for precise reference.

In [ ]:
# Extract records from each record set (identified by @id)
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records from RecordSet: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records and columns: {df.columns.tolist()}")

if record_set_id is not None:
    print(f"\nColumns of {record_set_id}")
    print(dataframes[record_set_id].columns.tolist())
    dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Data can be filtered, transformed, and grouped for analysis. Reference columns/fields by their precise Croissant `@id`.

- We'll select a numeric column (by its proper `@id`) for normalization and filtering.
- We'll demonstrate grouping by a categorical field (by Croissant `@id`), e.g. by participant gender or education if present.

In [ ]:
# --- Column selection by @id ---
# You must inspect the printed field @ids above and pick suitable ones. 
# For demonstration, let's detect likely numeric/categorical fields by their name.

df = dataframes[record_set_id]

# Try to select GAD-7, PHQ-9, or PSQ total scores (common mental health survey metrics), fallback to any numeric
possible_score_fields = [c for c in df.columns if any(x in c.lower() for x in ['gad', 'phq', 'psq', 'score', 'total'])]

if not possible_score_fields:
    # fallback: first field with integer/float dtype
    numeric_cols = df.select_dtypes(include=['number']).columns
    numeric_field = numeric_cols[0] if len(numeric_cols) else None
else:
    numeric_field = possible_score_fields[0]
print(f"Using numeric_field (for analysis): {numeric_field}")

threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head(3))

# Normalize the selected numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head(3))

# Try to group by a demographic field, e.g., gender or education
group_fields_candidates = [c for c in df.columns if any(x in c.lower() for x in ['gender', 'education', 'village', 'marital', 'age'])]
group_field = group_fields_candidates[0] if group_fields_candidates else None
if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped mean of {numeric_field} by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable group field detected for grouping.")

## 5. Visualization
Visualize distributions and group differences using matplotlib/seaborn. Reference all fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Distribution of the main numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=20)
plt.xlabel(numeric_field)
plt.title(f"Distribution of {numeric_field}")
plt.show()

# If group_field was found, show group-wise boxplot
if group_field is not None:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.xlabel(group_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion

- We loaded a FAIR-compliant survey dataset via Croissant, referencing all entities by their global `@id`.
- Programmatic exploration revealed key fields and provided a DataFrame per record set with correct, stable columns.
- Example analysis included filtering, normalization, and group comparison.
- Visualizations depicted score distributions and group differences for data quality and insight.

**Next steps:** Use the field `@id`s for reproducible, extensible analysis pipelines, and consult the dataset description fields within Croissant for metadata-driven workflows.